In [1]:
from pathlib import Path
import sys
import pandas as pd

sys.path.append(str(Path().resolve().parent))
from src.data_processing.data_processing import make_data_from_faculty, clean_data, make_data_from_program, make_features, process_data
from src.data_processing.data_processing import main as main_data

In [2]:
df_clean = main_data()

/mnt/d/ml_project/ml_project_2026/src/data_processing/data_processing.py:8: DtypeWarning: Columns (0: course, 1: module, 2: academic_year) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, sep = ';')


In [3]:
df_clean = clean_data(df_clean)
df_clean = process_data(df_clean)



In [4]:
df_clean['exam_type'].unique()

<StringArray>
[                     'Первая сдача',                         'Пересдача',
                  'Полный перезачет',             'Пересдача с комиссией',
 'Пересдача по уважительной причине',               'Частичный перезачет']
Length: 6, dtype: str

In [5]:
import statistics

def make_features_loc(df : pd.DataFrame) -> pd.DataFrame:
    result = []
    
    for student_id_hash, group in df.groupby('student_id_hash'):
        grades = group['grade_10'].tolist()
        avg_grade = sum(grades) / len(grades)

        
        student_status = group['student_status'].iloc[0]
        
        
        result.append({
            'student__id_hash': student_id_hash,
            'grades_list': grades,
            'avg_grade': avg_grade,
            'median_grade': statistics.median(grades),
            'min_grade': min(grades),
            'max_grade': max(grades),
            'count_grades': len(grades),
            # 'count_retake': len(group[group['exam_type'] == 'Пересдача']),
            'proportion_retake': len(group[group['exam_type'] == 'Пересдача'])/len(grades),
            # 'count_retake_com': len(group[group['exam_type'] == 'Пересдача с комиссией']),
            'proportion_retake_com': len(group[group['exam_type'] == 'Пересдача с комиссией'])/len(grades),
            'student_status': student_status

        })

    
    
    return pd.DataFrame(result)

df_encoded = make_features_loc(df_clean)

In [6]:
df_encoded['count_grades'].unique()
# print(df_encoded.loc[0].notna().sum())

# cols = [s for s in df_encoded.columns if '3_2' in s]


# df_encoded_part = df_encoded[cols]

array([ 24, 115,  14,  22,  16,  41,  18,  52,  12,  55,  15,  13,  21,
        58,   9,  50,  19,  32,  26,  17,  61,  35,  28,  40,  25,  34,
        11,   1,  65,  20,  38,  31,  10,   4,   8,  49,  39,   7,   6,
         3,  37,  62,  56,  51,  47,  29,  64,  57,  53,  33,  68,   5,
        27,   2,  42,  45,  59,  23,  54,  44,  43,  36,  63,  66,  30,
        60,  46,  98,  48,  70,  75,  91, 100,  69,  67,  72,  76,  77,
        74,  82,  73,  92,  71,  81, 109,  85, 107,  94, 126, 102,  93,
        96,  84,  90, 157, 121,  88, 149,  78,  80,  89,  83,  87,  95,
        79, 125, 118, 132, 154, 106, 104,  86,  97, 112, 111, 152, 114,
        99, 103, 131, 120, 113, 124, 108])

In [7]:
def str_to_int_stadent_status (df : pd.DataFrame):
    df["student_status"] = df["student_status"].map({
        "study": 0,
        "graduated": 1,
        "expelled": 2,
        "leave" : 3
    })
    return df

In [8]:
df_encoded = str_to_int_stadent_status(df_encoded)


In [9]:
df_encoded

,student__id_hash,grades_list,avg_grade,median_grade,min_grade,max_grade,count_grades,proportion_retake,proportion_retake_com,student_status
0,00000d86dad0d3c197341f776bb61a1d,"[10, 9, 10, 10, 10, 10, 7, 10, 8, 7, 8, 10, 10...",8.166667,9.0,2,10,24,0.000000,0.000000,1
1,0001dd443aff5c0800694730323c3d7d,"[8, 9, 7, 9, 8, 7, 9, 8, 7, 7, 8, 8, 7, 7, 7, ...",6.973913,7.0,0,10,115,0.034783,0.008696,0
2,00035b28bedc6d3a0336cb61a841d79b,"[8, 6, 5, 8, 6, 8, 9, 8, 7, 9, 7, 8, 8, 4]",7.214286,8.0,4,9,14,0.000000,0.000000,0
3,00044a704df898b9a87faf40795d62b6,"[8, 8, 9, 8, 9, 9, 8, 10, 10, 10, 7, 8, 9, 9, ...",8.454545,8.0,7,10,22,0.000000,0.000000,1
4,0004d03d5c41e468a521ed8a964fe889,"[6, 9, 8, 8, 7, 7, 9, 7, 8, 7, 9, 7, 8, 6, 3, 7]",7.250000,7.0,3,9,16,0.062500,0.000000,0
...,...,...,...,...,...,...,...,...,...,...
62925,fffaf96349866c4fbf94dca72aad6f0e,"[7, 7, 9, 7, 7, 4, 7, 7, 7, 7, 7, 4, 6, 6, 4, ...",6.709091,7.0,3,10,55,0.018182,0.000000,0
62926,fffbf91749bd6fd12d3ddf2ee81ee633,"[10, 8, 10, 7, 9, 8, 8, 9, 8, 8, 6, 8, 8, 8, 8...",8.238095,8.0,6,10,21,0.000000,0.000000,0
62927,fffc29ec23eda92a88df30e449afcf1c,"[6, 5, 7, 7, 10, 7, 8, 6, 6, 8, 8, 6, 4, 5, 7,...",6.520000,6.0,4,10,25,0.000000,0.000000,1
62928,fffe12fc2c520bcb03f1437defdc7ecf,"[8, 10, 9, 8, 8, 7, 7, 9, 10, 8, 7, 8, 8, 7, 6...",7.600000,8.0,5,10,25,0.000000,0.000000,1


In [25]:
df_encoded_2 = df_encoded[df_encoded['count_grades'] > 100]

In [26]:
df_encoded_2

,student__id_hash,grades_list,avg_grade,median_grade,min_grade,max_grade,count_grades,proportion_retake,proportion_retake_com,student_status
1,0001dd443aff5c0800694730323c3d7d,"[8, 9, 7, 9, 8, 7, 9, 8, 7, 7, 8, 8, 7, 7, 7, ...",6.973913,7.0,0,10,115,0.034783,0.008696,0
6551,1a74ee5ba73ee149df33f12804fe4850,"[8, 5, 4, 4, 5, 4, 4, 4, 3, 9, 4, 5, 3, 1, 7, ...",6.477064,7.0,0,10,109,0.055046,0.018349,0
9715,2747b9445ab10af1884d28377a8449c8,"[9, 8, 7, 7, 7, 8, 8, 7, 6, 7, 7, 7, 10, 8, 8,...",7.429907,7.0,5,10,107,0.000000,0.000000,0
10363,29d370dae5d04c60e0ac85b2076da264,"[7, 9, 10, 8, 7, 9, 5, 5, 8, 8, 4, 8, 9, 10, 4...",7.420561,8.0,0,10,107,0.018692,0.000000,0
10385,29e73c47fac6a477a28807c2ec0bcad4,"[7, 7, 5, 6, 6, 6, 8, 5, 7, 8, 7, 8, 6, 7, 5, ...",6.515873,6.5,0,10,126,0.023810,0.000000,0
10939,2c535b509956040160fbe78654cd7556,"[3, 4, 4, 5, 6, 5, 5, 6, 8, 3, 6, 5, 5, 5, 6, ...",5.656863,6.0,2,9,102,0.068627,0.000000,0
13884,385ba73871bb93cee1188276f6c1fcb1,"[5, 5, 4, 9, 5, 7, 6, 9, 2, 8, 5, 7, 4, 7, 8, ...",5.636943,6.0,0,10,157,0.050955,0.019108,0
17479,46abf2e7cd4dd4fe20a1d104345fa192,"[7, 7, 7, 7, 6, 8, 7, 7, 7, 8, 8, 8, 5, 6, 5, ...",5.669421,6.0,0,10,121,0.074380,0.016529,0
20330,51fa6fd4677884c4da2480982bb94c6e,"[8, 4, 10, 4, 6, 6, 8, 6, 7, 6, 9, 7, 6, 8, 9,...",7.080537,7.0,1,10,149,0.006711,0.006711,0
21605,5736ee0b807b8d32dc44d7ead9cbbdcb,"[10, 10, 10, 8, 9, 8, 8, 2, 8, 6, 9, 6, 8, 7, ...",7.880952,8.0,0,10,126,0.015873,0.000000,0


сделать тренд оценок (разница оценок)   
количество несдавших эту же дисциплину в его группе

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score

In [13]:
!pip show scikit-learn

Name: scikit-learn
Version: 1.8.0
Summary: A set of python modules for machine learning and data mining
Home-page: https://scikit-learn.org
Author: 
Author-email: 
License-Expression: BSD-3-Clause
Location: /mnt/d/ml_project/ml_project_2026/.venv/lib/python3.12/site-packages
Requires: joblib, numpy, scipy, threadpoolctl
Required-by: 


In [27]:
X = df_encoded.drop(columns=['student__id_hash', 'grades_list', 'student_status'])
y = df_encoded['student_status']



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)


model = LogisticRegression(solver='lbfgs', max_iter=10000)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)


print (accuracy_score(y_test, y_pred))

print(precision_score(y_test, y_pred, average='macro'))

0.6435722231050374
0.49094224881505527


/mnt/d/ml_project/ml_project_2026/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
